# Backpropagation

O algoritmo de **Backpropagation** (Retropropagação) é o motor principal responsável por treinar redes neurais profundas. Ele calcula o gradiente da função de erro (função de custo) em relação aos pesos da rede de forma extremamente eficiente, utilizando a **Regra da Cadeia** ("Chain Rule") do cálculo.

Neste notebook, abordaremos a intuição matemática e visual por trás do backpropagation e como a biblioteca PyTorch implementa esse processo automaticamente por meio do seu mecanismo de Autograd.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)

## Grafos Computacionais e a Regra da Cadeia

Para entender o backpropagation, primeiro precisamos visualizar as operações matemáticas como um **Grafo Computacional**.
Um grafo computacional é um grafo direcionado onde:
- Os nós (**nodes**) representam variáveis (entradas, pesos ou variáveis intermediárias).
- As arestas direcionadas indicam operações aplicadas sobre as variáveis, resultando na criação de novos nós.

Considere a função simples:
$$ f(x, y, z) = (x + y) \times z $$

Reescrevendo a equação introduzindo uma variável intermediária $q$:
1. $q = x + y$
2. $f = q \times z$

Queremos saber como os parâmetros originais ($x, y, z$) impactam nossa saída $f$. A Regra da Cadeia nos diz que:
$$ \frac{\partial f}{\partial x} = \frac{\partial f}{\partial q} \cdot \frac{\partial q}{\partial x} $$

Na nossa equação de exemplo:
- $\frac{\partial f}{\partial q} = z$ e $\frac{\partial q}{\partial x} = 1 \longrightarrow \frac{\partial f}{\partial x} = z \cdot 1 = z$
- Da mesma forma: $\frac{\partial f}{\partial y} = z \cdot 1 = z$
- $\frac{\partial f}{\partial z} = q = x + y$


## O Autograd do PyTorch

O mecanismo responsável por computar e gravar essas derivadas (gradientes) de maneira computacional e empilhável no PyTorch é o **Autograd**. 

In [ ]:
# Definindo nossas variáveis
x = torch.tensor([2.0], requires_grad=True)
y = torch.tensor([3.0], requires_grad=True)
z = torch.tensor([4.0], requires_grad=True)

# Forward pass
q = x + y
f = q * z

# Backward pass: calcula as derivadas parciais e preenche o atributo .grad
f.backward()

print("Gradientes computados via Autograd:")
print(f"x.grad (df/dx): {x.grad.item()}") # Esperado: z = 4.0
print(f"y.grad (df/dy): {y.grad.item()}") # Esperado: z = 4.0
print(f"z.grad (df/dz): {z.grad.item()}") # Esperado: q = 5.0

### Definindo os Tensores Folha com requires_grad
O processo começa com a definição dos tensores de entrada. Estes são as "folhas" do nosso grafo. Para que o PyTorch rastreie as operações e calcule os gradientes em relação a eles, devemos definir seu atributo `requires_grad` como `True`.

In [ ]:
# Definindo os tensores de entrada (folhas do grafo)
# Estes são os parâmetros em relação aos quais queremos os gradientes
a = torch.tensor(4.0, requires_grad=True)
b = torch.tensor(5.0, requires_grad=True)
c = torch.tensor(2.0, requires_grad=True)

print(f"Tensor 'a': {a}")
print(f"Tensor 'b': {b}")
print(f"Tensor 'c': {c}")

### Construindo o Grafo no Forward Pass
À medida que executamos as operações (o forward pass), o PyTorch constrói o grafo dinamicamente. Cada novo tensor resultante de uma operação armazena uma referência à função que o criou através do atributo `grad_fn`. Isso cria uma "história" computacional que permite a retropropagação.

In [ ]:
# Executando o forward pass para construir o grafo
z = a * b - c
L = z**2

print(f"Resultado intermediário 'z': {z}")
print(f"  - z.grad_fn: {z.grad_fn}\n") # z foi criado por uma subtração (SubBackward0)

print(f"Resultado final 'L': {L}")
print(f"  - L.grad_fn: {L.grad_fn}") # L foi criado por uma potenciação (PowBackward0)

### Calculando os Gradientes com .backward()
Com o grafo construído, podemos calcular os gradientes usando o método `.backward()` no tensor de saída final (que deve ser um escalar). Esta chamada inicia o processo de retropropagação a partir de $L$, aplicando a regra da cadeia para calcular as derivadas parciais de $L$ em relação a cada tensor folha.

Para o nosso exemplo, o autograd calcula:
$$ \frac{\partial L}{\partial a} = \frac{\partial L}{\partial z} \cdot \frac{\partial z}{\partial a} $$
E de forma análoga para $b$ e $c$.

In [ ]:
# Inicia a retropropagação a partir de L
L.backward()
print("O método L.backward() foi executado.")

### Acessando os Gradientes Calculados
Após a execução de `.backward()`, os gradientes calculados são acumulados no atributo `.grad` dos tensores folha (aqueles que foram inicializados com `requires_grad=True`).

In [ ]:
# Os gradientes agora estão disponíveis no atributo .grad de cada tensor folha
print(f"Gradiente dL/da: {a.grad}")
print(f"  - Cálculo manual: 2 * (4*5 - 2) * 5 = {2 * (4*5 - 2) * 5.0}\n")

print(f"Gradiente dL/db: {b.grad}")
print(f"  - Cálculo manual: 2 * (4*5 - 2) * 4 = {2 * (4*5 - 2) * 4.0}\n")

print(f"Gradiente dL/dc: {c.grad}")
print(f"  - Cálculo manual: 2 * (4*5 - 2) * (-1) = {2 * (4*5 - 2) * (-1.0)}")

### Desligando o Rastreamento
Para fases de inferência ou avaliação, onde não precisamos de gradientes, é crucial desativar o rastreamento para economizar memória e acelerar a computação. O gerenciador de contexto `with torch.no_grad():` instrui o PyTorch a não construir o grafo computacional para nenhuma operação dentro de seu escopo.


In [ ]:
print(f"Rastreamento de gradientes está ativado por padrão para 'a': {a.requires_grad}")

with torch.no_grad():
    # A mesma operação é executada aqui
    L_no_grad = (a * b - c)**2
    
    print(f"Resultado L_no_grad: {L_no_grad}")
    print(f"L_no_grad.requires_grad: {L_no_grad.requires_grad}")

print(f"O rastreamento de gradientes para 'a' continua ativado fora do bloco: {a.requires_grad}")

## Exemplo: Treinando um Neurônio

Vamos demonstrar como os gradientes alteram os parâmetros de um modelo (pesos e viés) rodando o backpropagation a cada passo.

A fórmula de atualização do Gradient Descent é:
$$ W = W - \alpha \cdot \frac{\partial L}{\partial W} $$

In [ ]:
# Entrada (X) e rótulo alvo (y_true)
X = torch.tensor([1.2, 0.5])
y_true = torch.tensor([1.0])

# Criando Pesos e Viés da camada
W = torch.randn(2, requires_grad=True)
b = torch.randn(1, requires_grad=True)

learning_rate = 0.1
losses = []

for epoch in range(5):
    # 1. Forward Pass
    logits = torch.dot(X, W) + b
    
    # 2. Computação da Perda (Mean Squared Error)
    loss = (logits - y_true)**2
    losses.append(loss.item())
    
    # 3. Backward Pass
    loss.backward()
    
    print(f"[Época {epoch}] Loss: {loss.item():.4f}")
    print(f"Gradiente em W (dL/dW): {W.grad.data}")
    print(f"Gradiente em b (dL/db): {b.grad.data}")
    print()
    
    # 4. Atualização de Pesos e Zerar Gradientes
    with torch.no_grad():
        W -= learning_rate * W.grad
        b -= learning_rate * b.grad
        
        W.grad.zero_()
        b.grad.zero_()

## Gradientes de uma MLP

Bibliotecas modernas como PyTorch facilitam imensamente o processo ao introduzirem as camadas (`nn.Linear`) e módulos como `nn.Sequential`. Nessas estruturas, os parâmetros (pesos e vieses) já estão encapsulados e configurados para computar gradiente.

Vamos criar uma MLP com duas camadas ocultas, uma de saída e injetar dados fictícios, verificando o fluxo dos gradientes pelo grafo encapsulado pelo `nn.Sequential`.

In [ ]:
# MLP simples com 3 camadas
mlp = nn.Sequential(
    nn.Linear(in_features=10, out_features=16),
    nn.ReLU(),
    nn.Linear(in_features=16, out_features=8),
    nn.ReLU(),
    nn.Linear(in_features=8, out_features=2)
)

print(mlp)

In [ ]:
# Simulando um "batch" com 5 amostras de teste (features de tamanho 10)
X_batch = torch.randn(5, 10)
y_batch_true = torch.tensor([0, 1, 1, 0, 1])

# Função de custo recomendada
criterion = nn.CrossEntropyLoss()

# Forward e cálculo do loss
logits = mlp(X_batch)
loss = criterion(logits, y_batch_true)

In [ ]:
print("Pesos da última camada ANTES do backward:")
print(mlp[4].weight.grad) 

# Propaga de volta pelos nós armazenados internamente pela MLP
loss.backward()

print("\nPesos da última camada DEPOIS do backward (dL/dW_final):")
print(mlp[4].weight.grad.shape)
print("Um espiada em alguns valores:")
print(mlp[4].weight.grad[:, :3]) 

print("\nTambém podemos ver os gradientes da primeira camada (dL/dW_hidden1):")
print(mlp[0].weight.grad.shape)